### 1) Метод Якоби и Ричардсона для решения СЛАУ(фул гпт, не интересно прогать это)

In [ ]:
import numpy as np
import numpy.linalg as la


def jacobi(A, b, x0=None, maxiter=10_000, tol=1e-8):
    A = np.asarray(A, float)
    b = np.asarray(b, float)
    n = A.shape[0]

    if x0 is None:
        x = np.zeros_like(b)
    else:
        x = np.asarray(x0, float).copy()

    # D – диагональ, R = A - D
    D = np.diag(A)
    if np.any(D == 0):
        raise ZeroDivisionError("Метод Якоби: нуль на диагонали A")

    R = A - np.diagflat(D)

    res_hist = []
    b_norm = la.norm(b)

    for k in range(maxiter):
        # x^{k+1} = (b - R x^k) / D (покомпонентно)
        x_new = (b - R @ x) / D

        # считаем невязку
        r = b - A @ x_new
        r_norm = la.norm(r)
        res_hist.append(r_norm)

        if r_norm <= tol * (b_norm if b_norm != 0 else 1.0):
            return x_new, k + 1, np.array(res_hist)

        x = x_new

    return x, maxiter, np.array(res_hist)


def richardson(A, b, x0=None, omega=0.1, maxiter=10_000, tol=1e-8, M=None):
    A = np.asarray(A, float)
    b = np.asarray(b, float)

    if x0 is None:
        x = np.zeros_like(b)
    else:
        x = np.asarray(x0, float).copy()

    n = A.shape[0]
    if M is None:
        # M = I, M^{-1}r = r
        def Minv(r):
            return r
    else:
        M = np.asarray(M, float)

        if np.allclose(M, np.diag(np.diag(M))):
            D = np.diag(M)

            def Minv(r):
                return r / D
        else:
            def Minv(r):
                return la.solve(M, r)

    res_hist = []
    b_norm = la.norm(b)

    for k in range(maxiter):
        r = b - A @ x
        r_norm = la.norm(r)
        res_hist.append(r_norm)

        if r_norm <= tol * (b_norm if b_norm != 0 else 1.0):
            return x, k + 1, np.array(res_hist)

        x = x + omega * Minv(r)

    return x, maxiter, np.array(res_hist)

##### Влияние обусловленности на сходимость Якоби

In [2]:
def make_spd_with_cond(n, kappa):
    """
    Строит SPD-матрицу A с условным числом ~ kappa.
    A = Q^T diag(lam) Q, где lam в [1, kappa].
    """
    Q, _ = la.qr(np.random.randn(n, n))
    lam = np.linspace(1.0, kappa, n)
    A = Q.T @ (lam[:, None] * Q)
    return A


if __name__ == "__main__":
    np.random.seed(0)
    n = 20
    b = np.ones(n)

    for kappa in [1e1, 1e2, 1e3, 1e4]:
        A = make_spd_with_cond(n, kappa)
        x, it, res = jacobi(A, b, tol=1e-6, maxiter=20_000)
        print(f"Jacobi: cond(A)≈{la.cond(A):.1e}, итераций={it}, финальная невязка={res[-1]:.1e}")

Jacobi: cond(A)≈1.0e+01, итераций=73, финальная невязка=4.2e-06
Jacobi: cond(A)≈1.0e+02, итераций=20000, финальная невязка=nan
Jacobi: cond(A)≈1.0e+03, итераций=5211, финальная невязка=4.5e-06
Jacobi: cond(A)≈1.0e+04, итераций=20000, финальная невязка=inf


/Users/anzhiday/Documents/vuichmatu/.venv/lib/python3.12/site-packages/numpy/linalg/_linalg.py:2792: RuntimeWarning: overflow encountered in dot
  sqnorm = x.dot(x)
/var/folders/w3/4l01d4cn60xc_qg78dq11vv40000gn/T/ipykernel_51350/1292757945.py:34: RuntimeWarning: overflow encountered in matmul
  r = b - A @ x_new
/var/folders/w3/4l01d4cn60xc_qg78dq11vv40000gn/T/ipykernel_51350/1292757945.py:31: RuntimeWarning: overflow encountered in matmul
  x_new = (b - R @ x) / D
/var/folders/w3/4l01d4cn60xc_qg78dq11vv40000gn/T/ipykernel_51350/1292757945.py:34: RuntimeWarning: invalid value encountered in matmul
  r = b - A @ x_new
/var/folders/w3/4l01d4cn60xc_qg78dq11vv40000gn/T/ipykernel_51350/1292757945.py:31: RuntimeWarning: invalid value encountered in matmul
  x_new = (b - R @ x) / D


##### Метод Ричардсона: влияние параметра $\omega$

In [3]:
if __name__ == "__main__":
    np.random.seed(1)
    n = 20
    A = make_spd_with_cond(n, 1e2)  # фиксируем A
    b = np.ones(n)

    for omega in [0.01, 0.05, 0.1, 0.3, 0.6]:
        x, it, res = richardson(A, b, omega=omega, tol=1e-6, maxiter=50_000)
        print(f"omega={omega:.2f}: итераций={it}, финальная невязка={res[-1]:.2e}")

omega=0.01: итераций=1304, финальная невязка=4.43e-06
omega=0.05: итераций=50000, финальная невязка=nan


/var/folders/w3/4l01d4cn60xc_qg78dq11vv40000gn/T/ipykernel_51350/1292757945.py:83: RuntimeWarning: overflow encountered in matmul
  r = b - A @ x
/var/folders/w3/4l01d4cn60xc_qg78dq11vv40000gn/T/ipykernel_51350/1292757945.py:83: RuntimeWarning: invalid value encountered in matmul
  r = b - A @ x
/var/folders/w3/4l01d4cn60xc_qg78dq11vv40000gn/T/ipykernel_51350/1292757945.py:90: RuntimeWarning: invalid value encountered in add
  x = x + omega * Minv(r)


omega=0.10: итераций=50000, финальная невязка=nan
omega=0.30: итераций=50000, финальная невязка=nan
omega=0.60: итераций=50000, финальная невязка=nan


##### Предобуславливатель в методе Ричардсона

In [4]:
if __name__ == "__main__":
    np.random.seed(2)
    n = 20
    A = make_spd_with_cond(n, 1e4)
    b = np.ones(n)
    M = np.diag(np.diag(A))  # диагональный предобуславливатель

    # Без предобуславливания
    x1, it1, res1 = richardson(A, b, omega=0.0005, tol=1e-6, maxiter=200_000, M=None)

    # С предобуславливанием
    x2, it2, res2 = richardson(A, b, omega=0.5, tol=1e-6, maxiter=200_000, M=M)

    print(f"Без предобуславливателя: итераций={it1}, финальная невязка={res1[-1]:.2e}")
    print(f"С  предобуславливателем: итераций={it2}, финальная невязка={res2[-1]:.2e}")

/var/folders/w3/4l01d4cn60xc_qg78dq11vv40000gn/T/ipykernel_51350/1292757945.py:83: RuntimeWarning: overflow encountered in matmul
  r = b - A @ x
/var/folders/w3/4l01d4cn60xc_qg78dq11vv40000gn/T/ipykernel_51350/1292757945.py:83: RuntimeWarning: invalid value encountered in matmul
  r = b - A @ x
/var/folders/w3/4l01d4cn60xc_qg78dq11vv40000gn/T/ipykernel_51350/1292757945.py:90: RuntimeWarning: invalid value encountered in add
  x = x + omega * Minv(r)


Без предобуславливателя: итераций=200000, финальная невязка=nan
С  предобуславливателем: итераций=91603, финальная невязка=4.47e-06


### 2) МНК с Тихоновской регуляризацией(тоже фул гпт не интересная хуйня)

In [9]:
import numpy as np
import numpy.linalg as la

def tikhonov_solve(X: np.ndarray, y: np.ndarray, lam: float) -> np.ndarray:
    """
    Solve (X^T X + lam I) w = X^T y  (Tikhonov / ridge).
    X: (n_samples, n_features)
    y: (n_samples,)
    lam: regularization strength λ >= 0
    """
    X = np.asarray(X, float)
    y = np.asarray(y, float)
    n_features = X.shape[1]

    XtX = X.T @ X
    A = XtX + lam * np.eye(n_features)
    b = X.T @ y

    w = la.solve(A, b)
    return w

def make_ill_conditioned_regression(n_samples=50, noise_std=0.1, seed=0):
    """
    Synthetic regression with highly correlated features (ill-conditioned X^T X).
    """
    rng = np.random.RandomState(seed)
    x = np.linspace(0, 1, n_samples)

    # features: 1, x, x^2, x^2 + 1e-4*x  -> почти линейная зависимость
    X = np.column_stack([
        np.ones_like(x),
        x,
        x**2,
        x**2 + 1e-4 * x
    ])

    w_true = np.array([1.0, 2.0, -3.0, 0.5])
    y_clean = X @ w_true
    y = y_clean + noise_std * rng.randn(n_samples)

    return X, y, w_true

import numpy as np
import numpy.linalg as la

if __name__ == "__main__":
    X, y, w_true = make_ill_conditioned_regression()
    n_features = X.shape[1]

    # "честное" МНК (λ = 0) через solve (без регуляризации)
    XtX = X.T @ X
    XtY = X.T @ y
    w_ls = la.solve(XtX, XtY)  # может быть нестабильным при сильной порче данных

    print("Условное число cond(X^T X) ≈", la.cond(XtX))

    lambdas = [0.0, 1e-6, 1e-4, 1e-2, 1.0, 10.0, 100.0]

    print("    λ    |||w_λ||_2 | train MSE | ||w_λ - w_true||_2")
    print("-" * 50)
    for lam in lambdas:
        w = tikhonov_solve(X, y, lam)

        y_pred = X @ w
        mse = np.mean((y_pred - y)**2)
        w_norm = la.norm(w)
        w_err = la.norm(w - w_true)

        print(f"{lam:8.1e} | {w_norm:8.3f} | {mse:9.4f} | {w_err:12.4f}")

Условное число cond(X^T X) ≈ 1.2093759759985125e+17
    λ    |||w_λ||_2 | train MSE | ||w_λ - w_true||_2
--------------------------------------------------
 0.0e+00 |   10.003 |    0.0109 |      12.1128
 1.0e-06 |    2.689 |    0.0109 |       2.4886
 1.0e-04 |    2.688 |    0.0109 |       2.4887
 1.0e-02 |    2.597 |    0.0109 |       2.5018
 1.0e+00 |    1.441 |    0.0272 |       3.2548
 1.0e+01 |    0.956 |    0.1200 |       3.5133
 1.0e+02 |    0.391 |    0.6192 |       3.6754


Когда запустишь код, обычно увидишь что-то в таком духе (цифры условные):
	•	при λ = 0:
	•	||w_λ|| очень большой (огромные по модулю коэффициенты),
	•	MSE маленькая (мы идеально подогнали шум),
	•	ошибка по весам ||w_λ - w_true|| большая → модель переобучилась, веса нестабильные;
	•	при маленьком λ (1e-6 … 1e-4):
	•	веса становятся более умеренными,
	•	MSE почти не растёт,
	•	||w_λ - w_true|| уменьшается → мы немного regularize и выигрываем по параметрам;
	•	при умеренном λ (1e-2, 1):
	•	веса ещё более «сглажены»,
	•	MSE на обучении немного растёт (потеряли точность аппроксимации),
	•	но ошибка параметров по отношению к w_true может быть минимальной → баланс bias/variance;
	•	при очень большом λ (10, 100):
	•	все коэффициенты сильно зажатые к нулю,
	•	MSE заметно растёт (модель недообучена),
	•	||w_λ - w_true|| снова растёт — мы слишком «перерегуляризовали».

### 3) Полиномиальная интерполяция в форме Лагранжа

Четвертое практическое задание
Реализовать на python следующие методы: метод Якоби и Ричардсона для решения СЛАУ (на тестах показать как сходимость зависит от обусловленности системы, как влияет итерационный параметр на сходимость для метода Ричардсона, на примерах показать как использование предобуславливателя влияет на сходимость), МНК с Тихоновской регуляризацией (проанализировать влияние регуляризации на тестах), Полиномиальная интерполяция в форме Лагранжа (провести исследование с ростом количества узлов для функции f(x) = 1/(1+25*x^2) на [-5, 5] на равномерной сетке и сетке Чебышева; реализовать, параллельный вариант для систем с общей памятью*), Сплайн-интерполяция кубическими сплайнами дефекта 1 (СЛАУ на моменты решать методом прогонки), метод трапеции и Симпсона для интегрирования (уметь численно определять порядок метод и строить график для него), метод Монте-Карло для интегрирования*. 

В качестве ответа прикрепить ноутбук и пдф (или иной читаемый в лмс скрипт)!

* -- не обязательно